# Assignment 12.1 - Graph Neural Networks



## Task 12.1.1: GNNs

* Implement a simple Graph Convolutional Network. **(RESULT)**
* Train and evaluate it on the Cora dataset provided below. **(RESULT)**

Hint: you might need to install torch_geometric first

`pip install torch_geometric`

In [ ]:
!pip -q install torch_geometric

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv

# Load Cora dataset (citation network: 2708 papers, 7 classes)
dataset = Planetoid(root='/tmp/Cora', name='Cora')
data = dataset[0]

Processing...
Done!


In [ ]:
# GCN Model + Training + Evaluation on Cora (RESULT)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data = data.to(device)

print("Using device:", device)
print("Num nodes:", data.num_nodes)
print("Num node features:", dataset.num_node_features)
print("Num classes:", dataset.num_classes)
print("Edges:", data.num_edges)
print("Train/Val/Test nodes:", int(data.train_mask.sum()), int(data.val_mask.sum()), int(data.test_mask.sum()))

class GCN(nn.Module):
    """
    Lecture-faithful 2-layer Graph Convolutional Network:
      H^(1) = ReLU( GCNConv(X, edge_index) )
      H^(2) = GCNConv(H^(1), edge_index)
    """
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.5):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x

hidden_dim = 16
dropout = 0.5
lr = 0.01
weight_decay = 5e-4
epochs = 200

model = GCN(
    in_channels=dataset.num_node_features,
    hidden_channels=hidden_dim,
    out_channels=dataset.num_classes,
    dropout=dropout
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
criterion = nn.CrossEntropyLoss()

@torch.no_grad()
def accuracy(mask):
    model.eval()
    logits = model(data.x, data.edge_index)
    pred = logits.argmax(dim=1)
    correct = (pred[mask] == data.y[mask]).sum().item()
    total = int(mask.sum())
    return correct / total

def train_one_epoch():
    model.train()
    optimizer.zero_grad(set_to_none=True)
    logits = model(data.x, data.edge_index)
    loss = criterion(logits[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return float(loss.item())

# Training loop (RESULT)
best_val = 0.0
best_test_at_best_val = 0.0
best_epoch = 0

for epoch in range(1, epochs + 1):
    loss = train_one_epoch()
    train_acc = accuracy(data.train_mask)
    val_acc = accuracy(data.val_mask)
    test_acc = accuracy(data.test_mask)

    if val_acc > best_val:
        best_val = val_acc
        best_test_at_best_val = test_acc
        best_epoch = epoch

    if epoch == 1 or epoch % 20 == 0:
        print(f"Epoch {epoch:03d}/{epochs} | loss {loss:.4f} | "
              f"train acc {train_acc*100:.2f}% | val acc {val_acc*100:.2f}% | test acc {test_acc*100:.2f}%")

print("\n================= RESULTS =================")
print(f"Best Val Acc: {best_val*100:.2f}% at epoch {best_epoch}")
print(f"Test Acc at Best Val: {best_test_at_best_val*100:.2f}%")
print("==========================================\n")

# RESULT
final_train = accuracy(data.train_mask)
final_val   = accuracy(data.val_mask)
final_test  = accuracy(data.test_mask)

print(f"Final Train Accuracy (RESULT): {final_train*100:.2f}%")
print(f"Final Val Accuracy   (RESULT): {final_val*100:.2f}%")
print(f"Final Test Accuracy  (RESULT): {final_test*100:.2f}%")


Using device: cpu
Num nodes: 2708
Num node features: 1433
Num classes: 7
Edges: 10556
Train/Val/Test nodes: 140 500 1000
Epoch 001/200 | loss 1.9433 | train acc 66.43% | val acc 39.60% | test acc 39.60%
Epoch 020/200 | loss 0.2964 | train acc 99.29% | val acc 77.00% | test acc 80.20%
Epoch 040/200 | loss 0.0552 | train acc 100.00% | val acc 77.80% | test acc 80.20%
Epoch 060/200 | loss 0.0724 | train acc 100.00% | val acc 78.60% | test acc 79.80%
Epoch 080/200 | loss 0.0414 | train acc 100.00% | val acc 78.00% | test acc 80.60%
Epoch 100/200 | loss 0.0350 | train acc 100.00% | val acc 78.80% | test acc 80.60%
Epoch 120/200 | loss 0.0437 | train acc 100.00% | val acc 78.00% | test acc 80.80%
Epoch 140/200 | loss 0.0285 | train acc 100.00% | val acc 78.00% | test acc 80.80%
Epoch 160/200 | loss 0.0440 | train acc 100.00% | val acc 77.60% | test acc 80.50%
Epoch 180/200 | loss 0.0306 | train acc 100.00% | val acc 77.60% | test acc 81.50%
Epoch 200/200 | loss 0.0267 | train acc 100.00% | v